# Texas Senate forecast

Rahan's working notebook for the Tier 1 Texas model (Decision 010). The calculations live in `src/model/` and `src/evaluate/`; this notebook imports them and walks through each step. `python -m src.model.run_texas` prints the same numbers from the repository root.

Run from `notebooks/` or the repository root after `pip install -r requirements.txt`.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src.model.polling_average import weighted_avg
from src.model.win_probability import p_dem_win
from src.model.run_texas import load_texas_lv, TEXAS_SNAPSHOT
from src.evaluate.calibration import (results_wide, polls_wide, merge_historical,
                                      calibration_errors, error_summary)

Load the 2026 Texas tracker snapshot, keep LV polls, and add D − R margins (Decision 009).

In [2]:
texas = load_texas_lv(TEXAS_SNAPSHOT)
texas[['poll_label', 'end_date', 'sample_size', 'dem_pct', 'rep_pct', 'margin']]

,poll_label,end_date,sample_size,dem_pct,rep_pct,margin
0,YouGov (in house),2026-10-05,3625,50.0,44.0,6.0
1,Fox News,2026-09-28,881,51.0,49.0,2.0
2,New York Times/Sienna,2026-09-30,615,49.0,43.0,6.0
4,TSU/YouGov,2026-09-19,1800,47.0,46.0,1.0
5,Emerson College/Nexstar,2026-09-14,1000,47.0,46.0,1.0
6,Recon MR,2026-09-11,614,49.0,43.0,6.0
7,NBC News/Mason-Dixon,2026-09-10,625,46.0,43.0,3.0
8,AARP Texas Voter Survey,2026-09-01,895,49.0,44.0,5.0
11,Emerson College,2026-08-10,1000,46.0,47.0,-1.0
12,Bush School/Recon MR,2026-07-30,619,47.0,43.0,4.0


Recency-weighted average: weight = 0.5^(age_days / h), with a working half-life of h = 14 days.

In [3]:
set_date = pd.to_datetime('2026-10-08')
h = 14  # poll half-life (days)

texas_avg, texas_neff = weighted_avg(texas, set_date, h)
texas_avg, texas_neff

(np.float64(3.9519617448017574), np.float64(6.705115664083077))

Historical results (MEDSL) and selected historical polls, one row per contest and per poll question.

In [4]:
wide = results_wide()
wide.to_csv(ROOT / 'outputs/results_wide.csv', index=True)
wide.head()

election,dem_votes,rep_votes,other_votes,valid_votes,dem_pct,rep_pct,margin
contest_id,,,,,,,
2018-AZ-ordinary-general,1191100,1135200,58008,2384308,49.955794,47.611299,2.344496
2018-CT-ordinary-general,825579,545717,15544,1386840,59.529506,39.349673,20.179833
2018-DE-ordinary-general,217385,137127,8080,362592,59.953060,37.818540,22.134520
2018-FL-ordinary-general,4089472,4099505,1028,8190005,49.932473,50.054976,-0.122503
2018-HI-ordinary-general,276316,112035,0,388351,71.151098,28.848902,42.302196


In [5]:
wide_polls = polls_wide()
wide_polls.to_csv(ROOT / 'outputs/polls_wide.csv', index=False)
wide_polls[['question_key', 'contest_id', 'end_date', 'dem_pct', 'rep_pct', 'margin']].head()

,question_key,contest_id,end_date,dem_pct,rep_pct,margin
0,538:2018:100:53712:86196,2018-FL-ordinary-general,2018-07-25,44.0,47.0,-3.0
1,538:2018:100:53714:86198,2018-FL-ordinary-general,2018-07-14,47.0,43.0,4.0
2,538:2018:100:53715:86682,2018-FL-ordinary-general,2018-06-22,41.0,46.0,-5.0
3,538:2018:100:53718:86202,2018-FL-ordinary-general,2018-06-15,50.0,40.0,10.0
4,538:2018:100:53725:86647,2018-FL-ordinary-general,2018-05-07,45.0,45.0,0.0


Merge historical polls and results

In [6]:
historical = merge_historical(wide_polls, wide)
historical[['contest_id', 'end_date', 'election_date', 'margin', 'margin_result']].head()

,contest_id,end_date,election_date,margin,margin_result
0,2018-FL-ordinary-general,2018-07-25,2018-11-06,-3.0,-0.122503
1,2018-FL-ordinary-general,2018-07-14,2018-11-06,4.0,-0.122503
2,2018-FL-ordinary-general,2018-06-22,2018-11-06,-5.0,-0.122503
3,2018-FL-ordinary-general,2018-06-15,2018-11-06,10.0,-0.122503
4,2018-FL-ordinary-general,2018-05-07,2018-11-06,0.0,-0.122503


Calibration loop: for each contest, compute the average at several horizons before Election Day and the error = actual margin − average. Contest-horizons with no polls yet are dropped.

In [7]:
errors, dropped = calibration_errors(historical, h)
dropped

,contest_id,horizon
0,2018-WY-ordinary-general,7
1,2018-WY-ordinary-general,14
2,2018-WY-ordinary-general,28
3,2018-WY-ordinary-general,42
4,2020-DE-ordinary-general,42
5,2020-OR-ordinary-general,42
6,2020-WV-ordinary-general,42
7,2020-WY-ordinary-general,7
8,2020-WY-ordinary-general,14
9,2020-WY-ordinary-general,28


Root mean squared error (RMSE) and mean error by horizon

In [8]:
by_horizon = error_summary(errors, 'horizon')
by_horizon

,races,mean_error,rmse
horizon,,,
7,50,-2.840304,6.495299
14,50,-3.162579,6.803297
28,50,-3.874173,7.029683
42,47,-3.102196,6.448441


RMSE and mean error by cycle across horizons

In [9]:
by_cycle = error_summary(errors, ['cycle', 'horizon'])
by_cycle

races  mean_error      rmse
cycle horizon                             
2018  7           27   -0.117469  4.921138
      14          27   -0.660559  5.430790
      28          27   -1.662637  5.136256
      42          27   -1.829807  5.828816
2020  7           22   -6.083294  8.063552
      14          22   -6.140938  8.232278
      28          22   -6.541015  8.878916
      42          19   -4.801294  7.291945
2024  7            1   -5.011092  5.011092
      14           1   -5.193236  5.193236
      28           1   -4.915122  4.915122
      42           1   -5.173805  5.173805

Quick outlier scan

In [10]:
errors.pivot_table(index='contest_id', columns='horizon', values='error')

horizon,7,14,28,42
contest_id,,,,
2018-AZ-ordinary-general,2.197009,3.480706,1.242379,-0.773174
2018-CT-ordinary-general,3.014048,4.957611,4.957611,3.179833
2018-DE-ordinary-general,-12.615316,-12.615316,-12.615316,-12.615316
2018-FL-ordinary-general,-2.086851,-2.767244,-0.931283,-1.488955
2018-IN-ordinary-general,-7.257128,-8.031676,-7.344913,-6.833249
2018-MA-ordinary-general,-0.472185,-1.973233,-1.973233,-3.348566
2018-MD-ordinary-general,-4.450637,-4.450637,-4.450637,-4.450637
2018-ME-ordinary-general,2.146032,-8.471247,-8.471247,-7.515624
2018-MI-ordinary-general,-5.775149,-7.756802,-12.028045,-12.211375


Set sigma to the closest horizon to today (manual now; automatic selection comes later). No bias shift (Decision 016).

In [11]:
sigma = by_horizon.loc[28, 'rmse']
sigma

np.float64(7.0296825989744365)

Calculate the probability of a Democratic win in Texas:

In [12]:
prob = p_dem_win(texas_avg, sigma)
print(f"Probability of a Democrat winning: {prob * 100:.2f}%")

Probability of a Democrat winning: 71.30%
